In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ipl_csv.zip to ipl_csv.zip


In [ ]:
import zipfile
import os

print("Uploaded files:", list(uploaded.keys()))

zip_filename = list(uploaded.keys())[0]
print("Zip file name:", zip_filename)


with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('ipl_data')

print("Unzipped successfully!")

files_inside = os.listdir('ipl_data')
print(f"\nTotal files found: {len(files_inside)}")
print("First 5 files:", files_inside[:5])

Uploaded files: ['ipl_csv.zip']
Zip file name: ipl_csv.zip
Unzipped successfully!

Total files found: 1244
First 5 files: ['598026.csv', '598069.csv', '392199.csv', '1136585.csv', '1529311.csv']


In [ ]:

for root, dirs, files_list in os.walk('ipl_data'):
    csv_files = [f for f in files_list if f.endswith('.csv')]
    if csv_files:
        print(f"CSV files found in: {root}")
        print(f"Count: {len(csv_files)}")
        print(f"Sample: {csv_files[:3]}")
        break

CSV files found in: ipl_data
Count: 1243
Sample: ['598026.csv', '598069.csv', '392199.csv']


In [ ]:
import pandas as pd
import glob
import os

FOLDER_PATH = 'ipl_data/'

all_files = glob.glob(FOLDER_PATH + '*.csv')
print(f"Total files: {len(all_files)}")

# Read ONE file correctly first
def read_cricsheet_file(filepath):
    # Read all rows
    df = pd.read_csv(
        filepath,
        header=None,           # no automatic header
        on_bad_lines='skip'    # skip any broken lines
    )

    # Keep only rows where first column = 'ball'
    # These are the actual delivery rows
    df = df[df[0] == 'ball']

    return df

# Test on one file
df_test = read_cricsheet_file(all_files[0])
print(df_test.head())
print(df_test.shape)


Total files: 1243
Empty DataFrame
Columns: [0, 1]
Index: []
(0, 2)


In [ ]:
with open(all_files[0], 'r') as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"\n=== First 30 lines raw ===")
for i, line in enumerate(lines[:30]):
    print(f"Line {i:02d}: {repr(line)}")

Total lines: 318

=== First 30 lines raw ===
Line 00: 'version,1.6.0\n'
Line 01: 'info,balls_per_over,6\n'
Line 02: 'info,team,Chennai Super Kings\n'
Line 03: 'info,team,Rajasthan Royals\n'
Line 04: 'info,gender,male\n'
Line 05: 'info,season,2013\n'
Line 06: 'info,date,2013/04/22\n'
Line 07: 'info,event,Indian Premier League\n'
Line 08: 'info,match_number,30\n'
Line 09: 'info,venue,"MA Chidambaram Stadium, Chepauk"\n'
Line 10: 'info,city,Chennai\n'
Line 11: 'info,toss_winner,Rajasthan Royals\n'
Line 12: 'info,toss_decision,bat\n'
Line 13: 'info,player_of_match,MEK Hussey\n'
Line 14: 'info,umpire,S Asnani\n'
Line 15: 'info,umpire,AK Chaudhary\n'
Line 16: 'info,reserve_umpire,K Srinivasan\n'
Line 17: 'info,tv_umpire,Asad Rauf\n'
Line 18: 'info,match_referee,RR Jadeja\n'
Line 19: 'info,winner,Chennai Super Kings\n'
Line 20: 'info,winner_wickets,5\n'
Line 21: 'info,player,Chennai Super Kings,M Vijay\n'
Line 22: 'info,player,Chennai Super Kings,MEK Hussey\n'
Line 23: 'info,player,Chennai Su

In [ ]:
# Find where ball data starts
print("=== Lines 50 to 80 ===")
for i, line in enumerate(lines[50:80], start=50):
    print(f"Line {i:02d}: {repr(line)}")

=== Lines 50 to 80 ===
Line 50: 'info,registry,people,DJ Bravo,87e562a9\n'
Line 51: 'info,registry,people,JO Holder,0f721006\n'
Line 52: 'info,registry,people,JP Faulkner,808f425a\n'
Line 53: 'info,registry,people,K Srinivasan,042a8b69\n'
Line 54: 'info,registry,people,KK Cooper,557153ca\n'
Line 55: 'info,registry,people,M Vijay,4b57e452\n'
Line 56: 'info,registry,people,MEK Hussey,48fd7349\n'
Line 57: 'info,registry,people,MM Sharma,759ac88f\n'
Line 58: 'info,registry,people,MS Dhoni,4a8a2e3b\n'
Line 59: 'info,registry,people,R Ashwin,495d42a5\n'
Line 60: 'info,registry,people,R Dravid,0184dc35\n'
Line 61: 'info,registry,people,R Shukla,de4b0555\n'
Line 62: 'info,registry,people,RA Jadeja,fe93fd9d\n'
Line 63: 'info,registry,people,RR Jadeja,053de19b\n'
Line 64: 'info,registry,people,S Asnani,b04524bc\n'
Line 65: 'info,registry,people,S Badrinath,3576e47e\n'
Line 66: 'info,registry,people,SK Raina,1dc12ab9\n'
Line 67: 'info,registry,people,SK Trivedi,3d7e087f\n'
Line 68: 'info,registry

In [ ]:
FOLDER_PATH = 'ipl_data/'

all_files = glob.glob(FOLDER_PATH + '*.csv')
print(f"Total files: {len(all_files)}")

def parse_cricsheet_file(filepath):

    # Storage for this match
    match_info = {}
    balls = []

    with open(filepath, 'r') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        parts = line.split(',')

        row_type = parts[0]

        # ── Extract match metadata from info rows ──
        if row_type == 'info':
            if len(parts) >= 3:
                key   = parts[1]
                value = ','.join(parts[2:]).strip('"')

                if key == 'season':       match_info['season']      = value
                if key == 'date':         match_info['match_date']  = value
                if key == 'venue':        match_info['venue']        = value
                if key == 'city':         match_info['city']         = value
                if key == 'winner':       match_info['winner']       = value
                if key == 'winner_runs':  match_info['winner_runs']  = value
                if key == 'winner_wickets': match_info['winner_wickets'] = value
                if key == 'toss_winner':  match_info['toss_winner']  = value
                if key == 'toss_decision': match_info['toss_decision'] = value
                if key == 'player_of_match': match_info['player_of_match'] = value
                if key == 'match_number': match_info['match_number'] = value


                if key == 'team':
                    if 'team1' not in match_info:
                        match_info['team1'] = value
                    else:
                        match_info['team2'] = value

        # ── Extract ball by ball data ──
        elif row_type == 'ball':
            try:
                ball_row = {
                    'innings':          parts[1],
                    'over_ball':        parts[2],
                    'batting_team':     parts[3],
                    'striker':          parts[4],
                    'non_striker':      parts[5],
                    'bowler':           parts[6],
                    'runs_off_bat':     int(parts[7])  if parts[7] != '' else 0,
                    'extras':           int(parts[8])  if parts[8] != '' else 0,
                    'wides':            int(parts[9])  if parts[9] != '' else 0,
                    'noballs':          int(parts[10]) if parts[10] != '' else 0,
                    'byes':             int(parts[11]) if parts[11] != '' else 0,
                    'legbyes':          int(parts[12]) if parts[12] != '' else 0,
                    'penalty':          int(parts[13]) if parts[13] != '' else 0,
                    'wicket_type':      parts[14].strip('"') if len(parts) > 14 else '',
                    'player_dismissed': parts[15].strip('"') if len(parts) > 15 else '',
                }
                # Add match info to every ball row
                ball_row.update(match_info)
                balls.append(ball_row)
            except Exception as e:
                pass  # skip any malformed ball line

    return balls


# ── LOOP THROUGH ALL 1243 FILES ──
all_balls = []
errors    = []

for i, filepath in enumerate(all_files):
    try:
        balls = parse_cricsheet_file(filepath)
        all_balls.extend(balls)
    except Exception as e:
        errors.append(filepath)

    # Progress update every 100 files
    if (i + 1) % 100 == 0:
        print(f"Processed {i+1}/{len(all_files)} files — {len(all_balls)} deliveries so far")

print(f"\n✅ Done!")
print(f"Total deliveries: {len(all_balls)}")
print(f"Files with errors: {len(errors)}")

# Convert to DataFrame
df_ipl = pd.DataFrame(all_balls)
print(f"\nShape: {df_ipl.shape}")
print(f"Columns: {df_ipl.columns.tolist()}")
df_ipl.head()

Total files: 1243
Processed 100/1243 files — 23968 deliveries so far
Processed 200/1243 files — 47664 deliveries so far
Processed 300/1243 files — 71511 deliveries so far
Processed 400/1243 files — 95337 deliveries so far
Processed 500/1243 files — 118908 deliveries so far
Processed 600/1243 files — 142833 deliveries so far
Processed 700/1243 files — 166693 deliveries so far
Processed 800/1243 files — 190626 deliveries so far
Processed 900/1243 files — 214285 deliveries so far
Processed 1000/1243 files — 237953 deliveries so far
Processed 1100/1243 files — 261520 deliveries so far
Processed 1200/1243 files — 285313 deliveries so far

✅ Done!
Total deliveries: 295732
Files with errors: 0

Shape: (295732, 28)
Columns: ['innings', 'over_ball', 'batting_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'team1', 'team2', 'season', 'match_date', 'match_number', 'venue', 'city', 'toss_winne

,innings,over_ball,batting_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,...,match_date,match_number,venue,city,toss_winner,toss_decision,player_of_match,winner,winner_wickets,winner_runs
0,1,0.1,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,4,0,0,0,...,2013/04/22,30,"MA Chidambaram Stadium, Chepauk",Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN
1,1,0.2,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,2013/04/22,30,"MA Chidambaram Stadium, Chepauk",Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN
2,1,0.3,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,2013/04/22,30,"MA Chidambaram Stadium, Chepauk",Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN
3,1,0.4,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,2013/04/22,30,"MA Chidambaram Stadium, Chepauk",Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN
4,1,0.5,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,1,0,0,0,...,2013/04/22,30,"MA Chidambaram Stadium, Chepauk",Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN


In [ ]:
# ── CLEAN DATA TYPES ──

# Fix date
df_ipl['match_date'] = pd.to_datetime(df_ipl['match_date'])

# Fix numeric columns
df_ipl['season']      = pd.to_numeric(df_ipl['season'],      errors='coerce')
df_ipl['match_number']= pd.to_numeric(df_ipl['match_number'],errors='coerce')
df_ipl['winner_runs'] = pd.to_numeric(df_ipl['winner_runs'], errors='coerce')

# Fill empty wicket columns with 'none'
df_ipl['wicket_type']       = df_ipl['wicket_type'].replace('', 'none')
df_ipl['player_dismissed']  = df_ipl['player_dismissed'].replace('', 'none')

# Add helper columns
df_ipl['total_runs'] = (
    df_ipl['runs_off_bat'] +
    df_ipl['wides']        +
    df_ipl['noballs']      +
    df_ipl['byes']         +
    df_ipl['legbyes']
)

df_ipl['is_wicket'] = df_ipl['wicket_type'].apply(
    lambda x: 0 if x == 'none' or x == '' else 1
)

# Add unique match_id from filename
df_ipl['match_id'] = df_ipl.groupby(
    ['season', 'match_number', 'team1', 'team2']
).ngroup()

print("Cleaning done!")
print(df_ipl.dtypes)
print(f"\nSeasons: {sorted(df_ipl['season'].dropna().unique().astype(int))}")
print(f"\nSample data:")
df_ipl.head(10)

Cleaning done!
innings                     object
over_ball                   object
batting_team                object
striker                     object
non_striker                 object
bowler                      object
runs_off_bat                 int64
extras                       int64
wides                        int64
noballs                      int64
byes                         int64
legbyes                      int64
penalty                      int64
wicket_type                 object
player_dismissed            object
team1                       object
team2                       object
season                     float64
match_date          datetime64[ns]
match_number               float64
venue                       object
city                        object
toss_winner                 object
toss_decision               object
player_of_match             object
winner                      object
winner_wickets              object
winner_runs                float64
total

,innings,over_ball,batting_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,...,city,toss_winner,toss_decision,player_of_match,winner,winner_wickets,winner_runs,total_runs,is_wicket,match_id
0,1,0.1,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,4,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,4,0,222.0
1,1,0.2,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,0,0,222.0
2,1,0.3,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,0,0,222.0
3,1,0.4,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,0,0,222.0
4,1,0.5,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,1,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,1,0,222.0
5,1,0.6,Rajasthan Royals,AM Rahane,SR Watson,MM Sharma,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,0,0,222.0
6,1,1.1,Rajasthan Royals,SR Watson,AM Rahane,JO Holder,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,0,0,222.0
7,1,1.2,Rajasthan Royals,SR Watson,AM Rahane,JO Holder,1,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,1,0,222.0
8,1,1.3,Rajasthan Royals,AM Rahane,SR Watson,JO Holder,1,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,1,0,222.0
9,1,1.4,Rajasthan Royals,SR Watson,AM Rahane,JO Holder,1,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,NaN,1,0,222.0


In [ ]:
# ── QUICK SANITY CHECK ──

print("=== Dataset Overview ===")
print(f"Total deliveries:  {len(df_ipl):,}")
print(f"Total matches:     {df_ipl['match_id'].nunique():,}")
print(f"Total seasons:     {df_ipl['season'].nunique()}")
print(f"Total players:     {df_ipl['striker'].nunique():,}")
print(f"Total teams:       {df_ipl['batting_team'].nunique()}")
print(f"\nSeasons covered: {sorted(df_ipl['season'].dropna().unique().astype(int))}")
print(f"\nMissing values:\n{df_ipl.isnull().sum()}")

=== Dataset Overview ===
Total deliveries:  295,732
Total matches:     1,002
Total seasons:     16
Total players:     739
Total teams:       19

Seasons covered: [np.int64(2009), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

Missing values:
innings                  0
over_ball                0
batting_team             0
striker                  0
non_striker              0
bowler                   0
runs_off_bat             0
extras                   0
wides                    0
noballs                  0
byes                     0
legbyes                  0
penalty                  0
wicket_type              0
player_dismissed         0
team1                    0
team2                    0
season               42546
match_date               0
match_number         17688
venue                   

In [ ]:
# Season is just the year of match_date
# So we can fill missing season from match_date directly
df_ipl['season'] = df_ipl['season'].fillna(
    df_ipl['match_date'].dt.year
)
print(f"Missing season: {df_ipl['season'].isna().sum()}")
print(f"Seasons: {sorted(df_ipl['season'].unique().astype(int))}")

Missing season: 0
Seasons: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [ ]:

# Older seasons didn't have match numbers — fill with 0
df_ipl['match_number'] = df_ipl['match_number'].fillna(0).astype(int)

print("After fixing match_number:")
print(f"Missing match_number: {df_ipl['match_number'].isna().sum()}")

After fixing match_number:
Missing match_number: 0


In [ ]:
# winner_runs and winner_wickets are BOTH supposed to be missing
# A match is won by EITHER runs OR wickets, never both
# So fill both with 0 — 0 means "not applicable for this match"
df_ipl['winner_runs']    = df_ipl['winner_runs'].fillna(0).astype(int)
df_ipl['winner_wickets'] = df_ipl['winner_wickets'].fillna(0).astype(int)

# winner missing means match had no result (abandoned/no result)
df_ipl['winner'] = df_ipl['winner'].fillna('No Result')

# player_of_match missing — fill with Unknown
df_ipl['player_of_match'] = df_ipl['player_of_match'].fillna('Unknown')

print("After fixing winner columns:")
print(f"Missing winner: {df_ipl['winner'].isna().sum()}")
print(f"Missing winner_runs: {df_ipl['winner_runs'].isna().sum()}")

After fixing winner columns:
Missing winner: 0
Missing winner_runs: 0


In [ ]:
# Now that season is fixed, rebuild match_id
df_ipl['match_id'] = df_ipl.groupby(
    ['season', 'match_date', 'team1', 'team2']
).ngroup() + 1   # start from 1 not 0

print("After rebuilding match_id:")
print(f"Missing match_id: {df_ipl['match_id'].isna().sum()}")
print(f"Total unique matches: {df_ipl['match_id'].nunique()}")

After rebuilding match_id:
Missing match_id: 0
Total unique matches: 1243


In [ ]:
df_ipl['season']      = df_ipl['season'].astype(int)
df_ipl['runs_off_bat']= df_ipl['runs_off_bat'].astype(int)
df_ipl['total_runs']  = df_ipl['total_runs'].astype(int)
df_ipl['is_wicket']   = df_ipl['is_wicket'].astype(int)

print("Data types fixed!")
print(df_ipl.dtypes)

Data types fixed!
innings                     object
over_ball                   object
batting_team                object
striker                     object
non_striker                 object
bowler                      object
runs_off_bat                 int64
extras                       int64
wides                        int64
noballs                      int64
byes                         int64
legbyes                      int64
penalty                      int64
wicket_type                 object
player_dismissed            object
team1                       object
team2                       object
season                       int64
match_date          datetime64[ns]
match_number                 int64
venue                       object
city                        object
toss_winner                 object
toss_decision               object
player_of_match             object
winner                      object
winner_wickets               int64
winner_runs                  int64
to

In [ ]:
print("=== FINAL DATASET OVERVIEW ===")
print(f"Total deliveries:  {len(df_ipl):,}")
print(f"Total matches:     {df_ipl['match_id'].nunique():,}")
print(f"Total seasons:     {df_ipl['season'].nunique()}")
print(f"Total players:     {df_ipl['striker'].nunique():,}")
print(f"Seasons: {sorted(df_ipl['season'].unique())}")
print(f"\nMissing values:\n{df_ipl.isnull().sum()}")
print(f"\nSample:")
df_ipl.head()

=== FINAL DATASET OVERVIEW ===
Total deliveries:  295,732
Total matches:     1,243
Total seasons:     19
Total players:     739
Seasons: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

Missing values:
innings             0
over_ball           0
batting_team        0
striker             0
non_striker         0
bowler              0
runs_off_bat        0
extras              0
wides               0
noballs             0
byes                0
legbyes             0
penalty             0
wicket_type         0
player_dismissed    0
team1               0
team2               0
season              0
match_date          0
match_number        0
venue               0
city                0
toss_winner         0
toss_decision       0
player_of_mat

,innings,over_ball,batting_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,...,city,toss_winner,toss_decision,player_of_match,winner,winner_wickets,winner_runs,total_runs,is_wicket,match_id
0,1,0.1,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,4,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,0,4,0,352
1,1,0.2,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,0,0,0,352
2,1,0.3,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,0,0,0,352
3,1,0.4,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,0,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,0,0,0,352
4,1,0.5,Rajasthan Royals,SR Watson,AM Rahane,MM Sharma,1,0,0,0,...,Chennai,Rajasthan Royals,bat,MEK Hussey,Chennai Super Kings,5,0,1,0,352


# **Table 1 : Matches**
**# One row per match**

In [ ]:
matches = df_ipl.groupby('match_id').agg(
    season          = ('season',          'first'),
    match_date      = ('match_date',      'first'),
    match_number    = ('match_number',    'first'),
    venue           = ('venue',           'first'),
    city            = ('city',            'first'),
    team1           = ('team1',           'first'),
    team2           = ('team2',           'first'),
    toss_winner     = ('toss_winner',     'first'),
    toss_decision   = ('toss_decision',   'first'),
    winner          = ('winner',          'first'),
    winner_runs     = ('winner_runs',     'first'),
    winner_wickets  = ('winner_wickets',  'first'),
    player_of_match = ('player_of_match', 'first'),
).reset_index()

print(f" Matches table: {matches.shape}")
print(matches.head())

 Matches table: (1243, 14)
   match_id  season match_date  match_number  \
0         1    2008 2008-04-18             1   
1         2    2008 2008-04-19             3   
2         3    2008 2008-04-19             2   
3         4    2008 2008-04-20             4   
4         5    2008 2008-04-20             5   

                                        venue        city  \
0                       M Chinnaswamy Stadium   Bangalore   
1                            Feroz Shah Kotla       Delhi   
2  Punjab Cricket Association Stadium, Mohali  Chandigarh   
3                                Eden Gardens     Kolkata   
4                            Wankhede Stadium      Mumbai   

                         team1                        team2  \
0  Royal Challengers Bangalore        Kolkata Knight Riders   
1             Delhi Daredevils             Rajasthan Royals   
2              Kings XI Punjab          Chennai Super Kings   
3        Kolkata Knight Riders              Deccan Chargers   
4 

# #Table 2 : **batting_stats**
**One row per player per match**

In [ ]:
batting_stats = df_ipl.groupby(
    ['match_id', 'batting_team', 'striker']
).agg(
    runs         = ('runs_off_bat', 'sum'),
    balls_faced  = ('runs_off_bat', 'count'),
    fours        = ('runs_off_bat', lambda x: (x == 4).sum()),
    sixes        = ('runs_off_bat', lambda x: (x == 6).sum()),
    is_out       = ('is_wicket',    'max')
).reset_index()

batting_stats.columns = [
    'match_id', 'team', 'player',
    'runs', 'balls_faced', 'fours', 'sixes', 'is_out'
]

batting_stats['strike_rate'] = round(
    batting_stats['runs'] * 100 /
    batting_stats['balls_faced'].replace(0, 1), 2
)

print(f"Batting stats table: {batting_stats.shape}")
print(batting_stats.head())

Batting stats table: (18768, 9)
   match_id                   team           player  runs  balls_faced  fours  \
0         1  Kolkata Knight Riders      BB McCullum   158           77     10   
1         1  Kolkata Knight Riders        DJ Hussey    12           12      1   
2         1  Kolkata Knight Riders  Mohammad Hafeez     5            3      1   
3         1  Kolkata Knight Riders       RT Ponting    20           20      1   
4         1  Kolkata Knight Riders       SC Ganguly    10           12      2   

   sixes  is_out  strike_rate  
0     13       0       205.19  
1      0       1       100.00  
2      0       0       166.67  
3      1       1       100.00  
4      0       1        83.33  


# Table 3 : bowling_stats
**one row per bowler per match**

In [ ]:
# bowling_team is simply the OTHER team in that match
# We know team1 and team2, and we know batting_team
# So bowling_team = whichever of team1/team2 is NOT the batting_team

df_ipl['bowling_team'] = df_ipl.apply(
    lambda row: row['team2'] if row['batting_team'] == row['team1'] else row['team1'],
    axis=1
)

# Verify it worked
print("Sample — batting vs bowling team:")
print(df_ipl[['batting_team', 'bowling_team', 'team1', 'team2']].head(10))
print(f"\nUnique bowling teams: {df_ipl['bowling_team'].nunique()}")

Sample — batting vs bowling team:
       batting_team         bowling_team                team1  \
0  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
1  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
2  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
3  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
4  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
5  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
6  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
7  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
8  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   
9  Rajasthan Royals  Chennai Super Kings  Chennai Super Kings   

              team2  
0  Rajasthan Royals  
1  Rajasthan Royals  
2  Rajasthan Royals  
3  Rajasthan Royals  
4  Rajasthan Royals  
5  Rajasthan Royals  
6  Rajasthan Royals  
7  Rajasthan Royals  
8  Rajasthan Royals  
9  Rajasthan Royals  

Unique 

In [ ]:
df_legal = df_ipl[(df_ipl['wides'] == 0) & (df_ipl['noballs'] == 0)].copy()

bowling_stats = df_ipl.groupby(
    ['match_id', 'bowling_team', 'bowler']
).agg(
    runs_given   = ('total_runs',  'sum'),
    wickets      = ('is_wicket',   'sum'),
    wides        = ('wides',       'sum'),
    noballs      = ('noballs',     'sum'),
).reset_index()

# Count legal balls per bowler per match
legal_balls = df_legal.groupby(
    ['match_id', 'bowler']
)['over_ball'].count().reset_index()
legal_balls.columns = ['match_id', 'bowler', 'legal_balls']

# Merge legal balls into bowling stats
bowling_stats = bowling_stats.merge(
    legal_balls,
    on=['match_id', 'bowler'],
    how='left'
)

bowling_stats['overs']   = round(bowling_stats['legal_balls'] / 6, 1)
bowling_stats['economy'] = round(
    bowling_stats['runs_given'] /
    bowling_stats['overs'].replace(0, 1), 2
)

bowling_stats.columns = [
    'match_id', 'bowling_team', 'bowler',
    'runs_given', 'wickets', 'wides', 'noballs',
    'legal_balls', 'overs', 'economy'
]

print(f"Bowling stats table: {bowling_stats.shape}")
print(bowling_stats.head())

Bowling stats table: (14700, 10)
   match_id           bowling_team      bowler  runs_given  wickets  wides  \
0         1  Kolkata Knight Riders  AB Agarkar          25        3      4   
1         1  Kolkata Knight Riders    AB Dinda           9        2      2   
2         1  Kolkata Knight Riders    I Sharma          13        1      1   
3         1  Kolkata Knight Riders   LR Shukla          12        1      3   
4         1  Kolkata Knight Riders  SC Ganguly          23        3      1   

   noballs  legal_balls  overs  economy  
0        0         24.0    4.0     6.25  
1        0         18.0    3.0     3.00  
2        0         18.0    3.0     4.33  
3        0          7.0    1.2    10.00  
4        0         24.0    4.0     5.75  


In [ ]:

print("=== 4 TABLES READY ===")
print(f"ball_by_ball:  {df_ipl.shape[0]:,} rows × {df_ipl.shape[1]} cols")
print(f"matches:       {matches.shape[0]:,} rows × {matches.shape[1]} cols")
print(f"batting_stats: {batting_stats.shape[0]:,} rows × {batting_stats.shape[1]} cols")
print(f"bowling_stats: {bowling_stats.shape[0]:,} rows × {bowling_stats.shape[1]} cols")

=== 4 TABLES READY ===
ball_by_ball:  295,732 rows × 32 cols
matches:       1,243 rows × 14 cols
batting_stats: 18,768 rows × 9 cols
bowling_stats: 14,700 rows × 10 cols


In [ ]:
import os
os.makedirs('ipl_tables', exist_ok=True)

# Save all 4 tables
df_ipl.to_csv('ipl_tables/ball_by_ball.csv', index=False)
matches.to_csv('ipl_tables/matches.csv', index=False)
batting_stats.to_csv('ipl_tables/batting_stats.csv', index=False)
bowling_stats.to_csv('ipl_tables/bowling_stats.csv', index=False)

print("All 4 CSVs saved!")
print(f"ball_by_ball.csv  — {len(df_ipl):,} rows")
print(f"matches.csv       — {len(matches):,} rows")
print(f"batting_stats.csv — {len(batting_stats):,} rows")
print(f"bowling_stats.csv — {len(bowling_stats):,} rows")

All 4 CSVs saved!
ball_by_ball.csv  — 295,732 rows
matches.csv       — 1,243 rows
batting_stats.csv — 18,768 rows
bowling_stats.csv — 14,700 rows


In [ ]:
# Filter to only 2025 and 2026 seasons
df_2526 = df_ipl[df_ipl['season'].isin([2025, 2026])].copy()


print(f"2025-2026 deliveries: {len(df_2526):,}")
print(f"Matches: {df_2526['match_id'].nunique()}")
print(f"Seasons: {df_2526['season'].unique()}")

2025-2026 deliveries: 34,812
Matches: 148
Seasons: [2026 2025]


In [ ]:
# Matches
matches_2526 = matches[matches['season'].isin([2025, 2026])].copy()

# Batting stats
batting_2526 = batting_stats[
    batting_stats['match_id'].isin(matches_2526['match_id'])
].copy()

# Bowling stats
bowling_2526 = bowling_stats[
    bowling_stats['match_id'].isin(matches_2526['match_id'])
].copy()

print("=== 2025-2026 Tables ===")
print(f"ball_by_ball:  {len(df_2526):,} rows")
print(f"matches:       {len(matches_2526):,} rows")
print(f"batting_stats: {len(batting_2526):,} rows")
print(f"bowling_stats: {len(bowling_2526):,} rows")

=== 2025-2026 Tables ===
ball_by_ball:  34,812 rows
matches:       148 rows
batting_stats: 2,253 rows
bowling_stats: 1,722 rows


In [ ]:

import os
os.makedirs('ipl_2526', exist_ok=True)

df_2526.to_csv('ipl_2526/ball_by_ball.csv',   index=False)
matches_2526.to_csv('ipl_2526/matches.csv',    index=False)
batting_2526.to_csv('ipl_2526/batting_stats.csv', index=False)
bowling_2526.to_csv('ipl_2526/bowling_stats.csv', index=False)

print("Saved!")

Saved!


In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile('ipl_2526.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('ipl_2526/ball_by_ball.csv',   'ball_by_ball.csv')
    zipf.write('ipl_2526/matches.csv',         'matches.csv')
    zipf.write('ipl_2526/batting_stats.csv',   'batting_stats.csv')
    zipf.write('ipl_2526/bowling_stats.csv',   'bowling_stats.csv')

print("Zip created!")
files.download('ipl_2526.zip')

Zip created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>